# Synthetic Natural dataset

In [1]:
import json
import random

## Loading lables, templates and samples

In [2]:
clinicalKnoledgeBase = json.load(open('./templates/clinicalKnowledgeBase.json', 'r'))
templates = json.load(open('./templates/templates.json', 'r'))
typos = json.load(open('./templates/typos.json', 'r'))

In [3]:
def NoiseInjection(text):
    words = text.split()

    noiseWords = [
        random.choice(typos.get(w.lower(), [w])) if random.random() < 0.65 else w
        for w in words
    ]

    return " ".join(noiseWords)

In [6]:
def generateDataset(numRecords):
    dataset = []
    consditions = list(clinicalKnoledgeBase.keys())

    for _ in range(numRecords):
        targetCondition = random.sample(consditions, k=random.choice([1, 2]))

        extractedSymptoms = []
        patientPhrases = []

        for condition in targetCondition:
            sympIndex = random.randint(0, len(clinicalKnoledgeBase[condition]["symptoms"]) - 1)
            extractedSymptoms.append(clinicalKnoledgeBase[condition]['symptoms'][sympIndex])
            patientPhrases.append(random.choice(clinicalKnoledgeBase[condition]['colloquial']))

        template = random.choice(templates)
        useesS2 = "{s2}" in template

        if useesS2:
            if len(patientPhrases)==1:
                sympIndex2 = (sympIndex + 1) % len(clinicalKnoledgeBase[targetCondition[0]]["symptoms"])
                extractedSymptoms.append(clinicalKnoledgeBase[targetCondition[0]]["symptoms"][sympIndex2])
                patientPhrases.append(random.choice(clinicalKnoledgeBase[targetCondition[0]]["colloquial"]))
            rawText = template.format(s1=patientPhrases[0], s2=patientPhrases[1])
        else:
            rawText = template.format(s1=patientPhrases[0])
        
        noiseText = NoiseInjection(rawText)

        record = {
            "patient_input": noiseText,
            "labels": {
                "symptoms_extracted": extractedSymptoms,
                "possible_conditions": targetCondition
            }
        }
        dataset.append(record)
    return dataset

In [7]:
syntheticDataset = generateDataset(1000)

In [8]:
for i, record in enumerate(syntheticDataset[:5]):
    print(f"[{i+1}] {record['patient_input']}")
    print(f"     Symptoms : {record['labels']['symptoms_extracted']}")
    print(f"     Condition: {record['labels']['possible_conditions']}")
    print()

[1] leaking fluid in my ear.
     Symptoms : ['Fluid discharge', 'Dizziness']
     Condition: ['Otitis Media (Middle Ear Infection)', 'Acoustic Neuroma']

[2] doc, im dealing with everything sounds muffled, plus my er is burning pain.
     Symptoms : ['Muffled hearing', 'Fluid discharge']
     Condition: ['Noise-Induced Hearing Loss', 'Otitis Media (Middle Ear Infection)']

[3] plugged.
     Symptoms : ['Muffled hearing']
     Condition: ['Eustachian Tube Dysfunction']

[4] when I wake up sudden dizziness and throughout the day partial deafness.
     Symptoms : ['Loss of balance', 'Ear blockage']
     Condition: ["Meniere's Disease", 'Cerumen Impaction (Ear Wax Blockage)']

[5] just ear pain, nothing else.
     Symptoms : ['Hearing loss', 'Facial numbness']
     Condition: ['Otitis Media (Middle Ear Infection)', 'Acoustic Neuroma']



In [9]:
with open('./dataset/audiology_dataset.json', 'w') as f:
    json.dump(syntheticDataset, f, indent=4)